# Aegis — Evaluation Notebook
### Golden Dataset • Judge Scoring • Metrics

In [ ]:
!pip install -q rich pandas matplotlib tqdm
import sys, os, json

In [ ]:
import zipfile

zip_path = "/mnt/data/Aegis-Self-Evolving-Agent-Fleet-main.zip"
extract_dir = "/kaggle/working/aegis_repo"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

sys.path.append("/kaggle/working/aegis_repo/Aegis-Self-Evolving-Agent-Fleet-main")

print("Repo extracted.")

In [ ]:
from services.orchestrator.orchestrator import Orchestrator
from services.agents.market_research_agent import MarketResearchAgent
from services.agents.copy_agent import CopyAgent
from services.agents.webdev_agent import WebDevAgent

orch = Orchestrator()
orch.register_agent("MarketResearchAgent", MarketResearchAgent(), "research")
orch.register_agent("CopyAgent", CopyAgent(), "copy")
orch.register_agent("WebDevAgent", WebDevAgent(), "deploy")

## Golden Dataset

In [ ]:
gd_path = "/kaggle/input/golden-dataset/golden_dataset.json"

if os.path.exists(gd_path):
    with open(gd_path) as f:
        golden = json.load(f)
else:
    golden = [
        {"id": 1, "mission": "Create a landing page for a smartwatch for students"},
        {"id": 2, "mission": "Launch a teaser campaign for a fitness app"},
        {"id": 3, "mission": "Summarize competitors for a Chennai food delivery startup"}
    ]


## Evaluate Missions

In [ ]:
from tqdm import tqdm
import pandas as pd

records=[]
for item in tqdm(golden):
    mission=item["mission"]
    out=orch.run_mission(mission)
    records.append({
        "id": item.get("id"),
        "mission": mission,
        "score": out["score"],
        "spans": len(out["trace"]),
        "session": out["session_id"]
})

df = pd.DataFrame(records)
df

## Metrics & Plot

In [ ]:
import matplotlib.pyplot as plt

print("Mean:", df["score"].mean())
print("Min:", df["score"].min())
print("Max:", df["score"].max())

plt.hist(df['score'], bins=[0,0.25,0.5,0.75,1.0], edgecolor='black')
plt.show()

## Regression Check

In [ ]:
THRESH=0.75
if df["score"].mean()<THRESH:
    print("Regression detected")
else:
    print("Stable")

## Inspect a session

In [ ]:
sid=df.loc[0,"session"]
session=orch.session_service.get_or_create(sid)
from rich.pretty import pprint
pprint(session.trace)

In [ ]:
df.to_csv("aegis_evaluation_results.csv", index=False)
"Saved aegis_evaluation_results.csv" 